In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Master Integrated Triage Pipeline Benchmark (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook integrates and benchmarks the complete **Multi-Stage Hierarchical Triage Architecture** on the **15% Holdout Test Set**, combining models from:
1. **[`models/xgboost_raw_esi1_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/xgboost_raw_esi1_extreme.ipynb)**: Layer 1 Binary ESI 1 Detector (XGBoost).
2. **[`models/rf_esi23_esi45_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/rf_esi23_esi45_extreme.ipynb)**: Layer 2A Grouped Class Model (Random Forest for ESI 2/3 vs 4/5 vs Other).
3. **[`models/xgboost_esi23_esi45_extreme.ipynb`](file:///home/apt2736/PKM_RF/models/xgboost_esi23_esi45_extreme.ipynb)**: Layer 2B Dual Specialist Model (LightGBM ESI 2/3 + XGBoost ESI 4/5).

### Mathematical Pipeline Inference & Probabilities
For any patient, the joint 5-class probability vector $\left[ P(\text{ESI 1}), P(\text{ESI 2}), P(\text{ESI 3}), P(\text{ESI 4}), P(\text{ESI 5}) \right]$ is calculated as:

$$\begin{aligned}
P_{\text{final}}(\text{ESI 1}) &= P_1(\text{ESI 1}) \\[6pt]
P_{\text{final}}(\text{ESI 2/3}) &= (1 - P_1(\text{ESI 1})) \times \frac{P_2(\text{ESI 2/3})}{P_2(\text{ESI 2/3}) + P_2(\text{ESI 4/5})} \\[6pt]
P_{\text{final}}(\text{ESI 4/5}) &= (1 - P_1(\text{ESI 1})) \times \frac{P_2(\text{ESI 4/5})}{P_2(\text{ESI 2/3}) + P_2(\text{ESI 4/5})}
\end{aligned}$$

### Benchmarking Metrics Suite
Evaluates full 5-class test performance using 5x5 Confusion Matrix, Accuracy, **Balanced Accuracy**, **Specificity**, Precision, Recall/Sensitivity, F1 Score, ROC-AUC, and **MCC Score**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(ranger)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Master Pipeline Benchmark Initialized ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Dataset & Construct Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
# Stratified 15% Holdout Test Partitioning
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
test_df      <- df_full[-in_train_val, ]
cat(sprintf("Holdout Test Set Ready: %d rows\n", nrow(test_df)))
cat("Natural 5-Class Target Distribution (ESI 1 to 5):\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load Saved Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
path_esi1 <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
path_rf   <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
path_ds   <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
cat("Loading model artifacts...\n")
art_esi1 <- readRDS(path_esi1)
art_rf   <- readRDS(path_rf)
art_ds   <- readRDS(path_ds)
cat("Model 1 (XGBoost ESI 1 Detector) Loaded.\n")
cat("Model 2A (Random Forest ESI 2/3 vs 4/5 vs Other) Loaded.\n")
cat("Model 2B (Dual Specialist LightGBM/XGBoost) Loaded.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Run Joint Hierarchical Pipeline Inference on Holdout Test Set
# ---------------------------------------------------------
# 1. Layer 1: XGBoost ESI 1 Detector Prediction
feats_esi1 <- setdiff(names(test_df), "target_col")
raw_feats_16 <- c("age", "gender", "cc_breathingdifficulty", "triage_vital_hr", "triage_vital_sbp", 
                  "triage_vital_rr", "triage_vital_o2", "pulse_last", "resp_last", "spo2_last", 
                  "sbp_last", "resp_min", "spo2_min", "sbp_min", "resp_max", "sbp_max")
test_esi1_scaled <- predict(art_esi1$preproc, test_df[, raw_feats_16])
dtest_esi1       <- xgb.DMatrix(data = as.matrix(test_esi1_scaled))
p_esi1           <- predict(art_esi1$model, dtest_esi1)
# 2. Layer 2A: Random Forest (ESI 2/3 vs 4/5 vs Other)
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols   <- setdiff(names(test_df), c(binary_cols, "target_col"))
test_rf_scaled <- predict(art_rf$preproc, test_df)
rf_raw_probs   <- predict(art_rf$model, data = test_rf_scaled)$predictions
p_rf_23 <- rf_raw_probs[, "2_3"]
p_rf_45 <- rf_raw_probs[, "4_5"]
denom_rf <- p_rf_23 + p_rf_45
denom_rf[denom_rf == 0] <- 1
p_rf_23_norm <- p_rf_23 / denom_rf
p_rf_45_norm <- p_rf_45 / denom_rf
# 3. Layer 2B: Dual Specialist (LightGBM 2/3 + XGBoost 4/5)
all_feats <- c(binary_cols, cont_cols)
test_ds_scaled <- predict(art_ds$preproc, test_df)
test_ds_x      <- as.matrix(test_ds_scaled[, all_feats])
if (art_ds$has_lgb) {
  p_ds_23 <- predict(art_ds$model_lgb, test_ds_x)
} else {
  p_ds_23 <- predict(art_ds$model_lgb, xgb.DMatrix(data = test_ds_x))
}
p_ds_45 <- predict(art_ds$model_xgb, xgb.DMatrix(data = test_ds_x))
denom_ds <- p_ds_23 + p_ds_45
denom_ds[denom_ds == 0] <- 1
p_ds_23_norm <- p_ds_23 / denom_ds
p_ds_45_norm <- p_ds_45 / denom_ds
# COMBINED PIPELINE A: Layer 1 XGBoost + Layer 2A Random Forest
p_pipeA_esi1 <- p_esi1
p_pipeA_23   <- (1 - p_esi1) * p_rf_23_norm
p_pipeA_45   <- (1 - p_esi1) * p_rf_45_norm
# COMBINED PIPELINE B: Layer 1 XGBoost + Layer 2B Dual Specialist (LightGBM/XGBoost)
p_pipeB_esi1 <- p_esi1
p_pipeB_23   <- (1 - p_esi1) * p_ds_23_norm
p_pipeB_45   <- (1 - p_esi1) * p_ds_45_norm
cat("Inference execution complete across all test samples.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark Pipeline A (Layer 1 XGBoost + Layer 2A Random Forest)
# ---------------------------------------------------------
calc_mcc <- function(tp, fp, fn, tn) {
  num <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
}
pipeA_mat <- cbind(p_pipeA_esi1, p_pipeA_23, p_pipeA_45)
colnames(pipeA_mat) <- c("1", "2_3", "4_5")
pipeA_pred_idx <- apply(pipeA_mat, 1, which.max)
pipeA_pred_fac <- factor(colnames(pipeA_mat)[pipeA_pred_idx], levels = c("1", "2_3", "4_5"))
act_grouped <- ifelse(test_df$target_col == "1", "1", ifelse(test_df$target_col %in% c("2", "3"), "2_3", "4_5"))
act_grouped_fac <- factor(act_grouped, levels = c("1", "2_3", "4_5"))
cm_A  <- confusionMatrix(pipeA_pred_fac, act_grouped_fac)
acc_A <- as.numeric(cm_A$overall["Accuracy"])
prec_A    <- as.numeric(cm_A$byClass[, "Pos Pred Value"])
rec_A     <- as.numeric(cm_A$byClass[, "Sensitivity"])
spec_A    <- as.numeric(cm_A$byClass[, "Specificity"])
bal_acc_A <- as.numeric(cm_A$byClass[, "Balanced Accuracy"])
prec_A[is.na(prec_A)]       <- 0
rec_A[is.na(rec_A)]         <- 0
spec_A[is.na(spec_A)]       <- 0
bal_acc_A[is.na(bal_acc_A)] <- 0
f1_A <- ifelse((prec_A + rec_A) > 0, 2 * (prec_A * rec_A) / (prec_A + rec_A), 0)
mcc_A <- sapply(1:3, function(i) {
  cls <- levels(act_grouped_fac)[i]
  tp  <- sum(pipeA_pred_fac == cls & act_grouped_fac == cls)
  tn  <- sum(pipeA_pred_fac != cls & act_grouped_fac != cls)
  fp  <- sum(pipeA_pred_fac == cls & act_grouped_fac != cls)
  fn  <- sum(pipeA_pred_fac != cls & act_grouped_fac == cls)
  calc_mcc(tp, fp, fn, tn)
})
cat("============================================================\n")
cat("   PIPELINE A BENCHMARK: LAYER 1 XGBOOST + LAYER 2A RANDOM FOREST\n")
cat("============================================================\n")
cat(sprintf("  Accuracy           : %.4f (%.2f%%)\n", acc_A, acc_A * 100))
cat(sprintf("  Macro Balanced Acc : %.4f\n", mean(bal_acc_A)))
cat(sprintf("  Macro Specificity  : %.4f\n", mean(spec_A)))
cat(sprintf("  Macro Precision    : %.4f\n", mean(prec_A)))
cat(sprintf("  Macro Recall (Sens): %.4f\n", mean(rec_A)))
cat(sprintf("  Macro F1-Score     : %.4f\n", mean(f1_A)))
cat(sprintf("  Macro MCC Score    : %.4f\n", mean(mcc_A)))
cat("============================================================\n\n")
print(cm_A$table)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Benchmark Pipeline B (Layer 1 XGBoost + Layer 2B Dual Specialist LightGBM/XGBoost)
# ---------------------------------------------------------
pipeB_mat <- cbind(p_pipeB_esi1, p_pipeB_23, p_pipeB_45)
colnames(pipeB_mat) <- c("1", "2_3", "4_5")
pipeB_pred_idx <- apply(pipeB_mat, 1, which.max)
pipeB_pred_fac <- factor(colnames(pipeB_mat)[pipeB_pred_idx], levels = c("1", "2_3", "4_5"))
cm_B  <- confusionMatrix(pipeB_pred_fac, act_grouped_fac)
acc_B <- as.numeric(cm_B$overall["Accuracy"])
prec_B    <- as.numeric(cm_B$byClass[, "Pos Pred Value"])
rec_B     <- as.numeric(cm_B$byClass[, "Sensitivity"])
spec_B    <- as.numeric(cm_B$byClass[, "Specificity"])
bal_acc_B <- as.numeric(cm_B$byClass[, "Balanced Accuracy"])
prec_B[is.na(prec_B)]       <- 0
rec_B[is.na(rec_B)]         <- 0
spec_B[is.na(spec_B)]       <- 0
bal_acc_B[is.na(bal_acc_B)] <- 0
f1_B <- ifelse((prec_B + rec_B) > 0, 2 * (prec_B * rec_B) / (prec_B + rec_B), 0)
mcc_B <- sapply(1:3, function(i) {
  cls <- levels(act_grouped_fac)[i]
  tp  <- sum(pipeB_pred_fac == cls & act_grouped_fac == cls)
  tn  <- sum(pipeB_pred_fac != cls & act_grouped_fac != cls)
  fp  <- sum(pipeB_pred_fac == cls & act_grouped_fac != cls)
  fn  <- sum(pipeB_pred_fac != cls & act_grouped_fac == cls)
  calc_mcc(tp, fp, fn, tn)
})
cat("============================================================\n")
cat("   PIPELINE B BENCHMARK: LAYER 1 XGBOOST + LAYER 2B DUAL SPECIALIST\n")
cat("============================================================\n")
cat(sprintf("  Accuracy           : %.4f (%.2f%%)\n", acc_B, acc_B * 100))
cat(sprintf("  Macro Balanced Acc : %.4f\n", mean(bal_acc_B)))
cat(sprintf("  Macro Specificity  : %.4f\n", mean(spec_B)))
cat(sprintf("  Macro Precision    : %.4f\n", mean(prec_B)))
cat(sprintf("  Macro Recall (Sens): %.4f\n", mean(rec_B)))
cat(sprintf("  Macro F1-Score     : %.4f\n", mean(f1_B)))
cat(sprintf("  Macro MCC Score    : %.4f\n", mean(mcc_B)))
cat("============================================================\n\n")
print(cm_B$table)
# Write CSV Summary Report
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
pipeline_report_df <- data.frame(
  Pipeline = c("Pipeline_A_XGB_RF", "Pipeline_B_XGB_LGB_XGB"),
  Accuracy = round(c(acc_A, acc_B), 4),
  Macro_Balanced_Accuracy = round(c(mean(bal_acc_A), mean(bal_acc_B)), 4),
  Macro_Specificity = round(c(mean(spec_A), mean(spec_B)), 4),
  Macro_Precision = round(c(mean(prec_A), mean(prec_B)), 4),
  Macro_Recall = round(c(mean(rec_A), mean(rec_B)), 4),
  Macro_F1_Score = round(c(mean(f1_A), mean(f1_B)), 4),
  Macro_MCC_Score = round(c(mean(mcc_A), mean(mcc_B)), 4)
)
write.csv(pipeline_report_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("Combined Pipeline Test Set Report written to: reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Diagnostic Plots (Metrics Bar Chart Comparing Pipeline A vs Pipeline B)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
metrics_comp <- data.frame(
  Metric = rep(c("Accuracy", "Balanced_Acc", "Specificity", "Precision", "Recall", "F1_Score", "MCC_Score"), 2),
  Pipeline = c(rep("Pipeline A (XGB + RF)", 7), rep("Pipeline B (XGB + Dual Specialist)", 7)),
  Score = c(
    acc_A, mean(bal_acc_A), mean(spec_A), mean(prec_A), mean(rec_A), mean(f1_A), mean(mcc_A),
    acc_B, mean(bal_acc_B), mean(spec_B), mean(prec_B), mean(rec_B), mean(f1_B), mean(mcc_B)
  )
)
p_bar <- ggplot(metrics_comp, aes(x = Metric, y = Score, fill = Pipeline)) +
  geom_bar(stat = "identity", position = "dodge", width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.6), vjust = -0.3, size = 3.0, fontface = "bold") +
  theme_minimal() +
  scale_fill_manual(values = c("Pipeline A (XGB + RF)" = "#2b5c8f", "Pipeline B (XGB + Dual Specialist)" = "#e07a5f")) +
  labs(title = "Full System Triage Pipeline Benchmark (15% Holdout Test Set)",
       subtitle = "Comparing Pipeline A (XGBoost ESI 1 + Random Forest) vs. Pipeline B (XGBoost ESI 1 + LightGBM/XGBoost Dual Specialist)",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 9.5, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "top")
ggsave(file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p_bar, width = 11, height = 5.5, dpi = 300)
cat("Combined Pipeline Comparison Bar Chart saved to: plots/combined_pipeline_metrics_barchart.png\n")
p_bar